In [0]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report

from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

In [0]:
RANDOM_STATE = 28082026

In [0]:
# from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier

In [0]:
import ast
from sklearn.calibration import CalibratedClassifierCV

# Load the Datasets

In [0]:
df = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_dataset_CLEAN.csv')

ex = pd.read_csv('/Volumes/projects/default/exercise-analysis/programs_detailed_boostcamp_kaggle_EDITED.csv')

# Exact-match bootstrap + Tfidf + SVM

In [0]:
def norm(s):
    s = str(s).lower()
    s = re.sub(r'\(.*?\)', '', s)          # strip equipment notes in parens
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

In [0]:
labeled = df
labeled['norm'] = labeled['exercise'].apply(norm)

prog = ex
unique_names = pd.DataFrame({'exercise_name': prog['exercise_name'].unique()})
unique_names['norm'] = unique_names['exercise_name'].apply(norm)

In [0]:
# Layer 1: Exact-match bootstrap
lookup = labeled.drop_duplicates('norm').set_index('norm')['label']

unique_names['label'] = unique_names['norm'].map(lookup)
to_predict = unique_names[unique_names['label'].isna()].copy()
print(f"{unique_names['label'].notna().sum()} matched directly, {len(to_predict)} need prediction")

In [0]:
# Layer 2:Tfidf + SVM Classifier
vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1,2), min_df=1, stop_words="english")),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=1)),
])
pipe = Pipeline([("feats", vectorizer), ("clf", LinearSVC(class_weight="balanced", max_iter=5000))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, labeled['exercise'], labeled['label'], cv=cv, scoring='f1_macro', n_jobs=-1)
print("CV macro-F1:", scores.mean(), scores.std())

pipe.fit(labeled['exercise'], labeled['label'])
to_predict['label'] = pipe.predict(to_predict['exercise_name'])

## Probabilities to estimate errors

In [0]:
calibrated = CalibratedClassifierCV(pipe, cv=5, method='sigmoid')
calibrated.fit(labeled['exercise'], labeled['label'])

probs = calibrated.predict_proba(to_predict['exercise_name'])
to_predict['confidence'] = probs.max(axis=1)
to_predict.sort_values('confidence').head(30)  # review these first


# Random Forest Approach

In [0]:
# Layer 2:Tfidf + SVM Classifier
vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1,2), min_df=1, stop_words="english")),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=1)),
])

In [0]:
model = {"RandomForest": RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )}

In [0]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([("feats", vectorizer), ("classifier", model['RandomForest'])])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, labeled['exercise'], labeled['label'], cv=cv, scoring='f1_macro', n_jobs=-1)
print("CV macro-F1:", scores.mean(), scores.std())

pipe.fit(labeled['exercise'], labeled['label'])
to_predict['label'] = pipe.predict(to_predict['exercise_name'])

In [0]:
scores

# XGBoost

# LightGBM